In [ ]:
print("START")

from LabData import config_global as config
from LabUtils import Utils
from LabUtils import addloglevels
import os
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import re
from sklearn.preprocessing import StandardScaler
import warnings

# ============================================================
# USER SETTINGS
# ============================================================
foods_only = True  # True: only foods (plus age/sex); False: all diet features
suffix_foods = "_foods_only" if foods_only else ""

PHENOTYPE = "bt__triglycerides"  # e.g. 'bt__triglycerides'
NEGATIVE_CONTROL = True          # <<< SET TRUE for Figure 5 negative control
# ============================================================


def main(q):
    home_path = "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/"
    SPECIES = "segal_species"  # 'mpa_species' or 'segal_species'
    PROBLEM = "regression"

    # -----------------------------
    # Load phenotype + microbiome table (for phenotype list / mb feature names)
    # -----------------------------
    phenotypes_mb = pd.read_pickle(
        "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/phenotypes_mb.pkl"
    )

    with open(
        "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/target_phenotypes.pkl", "rb"
    ) as f:
        target_phenotypes = pickle.load(f)

    flat_targets = sum(target_phenotypes, [])
    microbial_features = [c for c in phenotypes_mb.columns if c not in flat_targets + ["RegistrationCode"]]

    # -----------------------------
    # Load diet+microbiome (train/test defined elsewhere)
    # -----------------------------
    diet_mb = pd.read_pickle(home_path + f"data/{SPECIES}/diet_mb.pkl")
    diet_mb_test = pd.read_pickle(home_path + f"data/{SPECIES}/diet_mb_baseline_test.pkl")
    test_subjects = diet_mb_test.index

    diet_mb = diet_mb.loc[test_subjects, :]

    with open(home_path + f"data/{SPECIES}/my_lists.pkl", "rb") as file:
        loaded_lists = pickle.load(file)
    base_features, all_features, targets = loaded_lists

    all_features_formatted = all_features  # already sanitized in your pipeline

    # (Unused in this script, but keeping since you had it)
    with open(home_path + f"data/{SPECIES}/scaler.pkl", "rb") as scaler_file:
        _scaler_unused = pickle.load(scaler_file)

    # -----------------------------
    # Filter significant targets (taxa) for diet->microbiome mapping (as you already do)
    # -----------------------------
    with open(
        f"/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/{PROBLEM}/{SPECIES}/significant_targets.pkl",
        "rb",
    ) as file:
        significant_targets = pickle.load(file)
    significant_targets_indices = [targets.index(item) for item in significant_targets]

    with open("/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/food_shortnames.pkl", "rb") as file:
        food_shortnames = pickle.load(file)

    significant_targets_df = pd.DataFrame(columns=significant_targets)

    mb_names = pd.read_pickle("/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/mb_names.pkl")

    # -----------------------------
    # Restrict features to foods_only if requested
    # -----------------------------
    if foods_only:
        all_features = [f for f in all_features if f in ["age", "sex"] + list(food_shortnames)]
        all_features_formatted = all_features
        print("Len all features:", len(all_features))
        print("Len all features formatted:", len(all_features_formatted))

    DIET_ONLY_FEATURES = [f for f in all_features_formatted if f not in ("age", "sex")]

    # -----------------------------
    # Helpers: rename taxa to nicer names
    # -----------------------------
    def rename_microbiome_columns(diet_mb_scaled: pd.DataFrame, mb_names_df: pd.DataFrame, targets_list: list) -> pd.DataFrame:
        mb_names_df = mb_names_df.copy()
        mb_names_df.index = mb_names_df.index.astype(str).str.strip()

        species_map = mb_names_df["species_new"].astype(str).str.strip()
        genus_map = mb_names_df["genus_new"].astype(str).str.strip()
        family_map = mb_names_df["family_new"].astype(str).str.strip()

        final_mapping = {}
        for col in targets_list:
            name = species_map.get(col, None)
            if name in ("unknown", "nan") or pd.isna(name):
                name = genus_map.get(col, None)
            if name in ("unknown", "nan") or pd.isna(name):
                name = family_map.get(col, None)
            if name is None or name == "unknown" or pd.isna(name):
                name = col
            final_mapping[col] = name

        df2 = diet_mb_scaled.copy()
        df2.rename(columns=final_mapping, inplace=True)

        # de-duplicate
        col_counts = Counter(df2.columns)
        name_counter = defaultdict(int)
        new_cols = []
        for c in df2.columns:
            if col_counts[c] > 1:
                name_counter[c] += 1
                new_cols.append(f"{c}_{name_counter[c]}")
            else:
                new_cols.append(c)
        df2.columns = new_cols
        return df2

    microbial_features_df = rename_microbiome_columns(diet_mb[targets], mb_names, targets)
    significant_targets_df = rename_microbiome_columns(significant_targets_df, mb_names, targets)

    # -----------------------------
    # Load SHAP matrices used by your intervention logic
    # -----------------------------
    directional_phenotypes_shap = pd.read_pickle(
        "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/diet_intervention/directional_phenotypes_shap.pkl"
    )
    directional_phenotypes_shap = directional_phenotypes_shap[~directional_phenotypes_shap.index.isin(["age", "sex"])]
    directional_phenotypes_shap = directional_phenotypes_shap.iloc[significant_targets_indices, :]

    directional_microbiome_shap = pd.read_pickle(
        "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/diet_intervention/directional_microbiome_shap.pkl"
    )
    directional_microbiome_shap = directional_microbiome_shap[~directional_microbiome_shap.index.isin(["age", "sex"])]
    directional_microbiome_shap = directional_microbiome_shap.loc[food_shortnames, :]

    directional_triglycerides_shap = directional_phenotypes_shap.loc[:, PHENOTYPE]
    _ = directional_triglycerides_shap.abs().sort_values(ascending=False)

    # -----------------------------
    # Food bounds + NOVA
    # -----------------------------
    diet_foods_df = pd.read_csv(
        "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/diet_adherence_foods.csv",
        index_col=0,
    )
    diet_foods_df = diet_foods_df.loc[:, diet_foods_df.columns.isin(food_shortnames)]
    nova_foods_df = diet_foods_df.loc["NOVA", :]

    positive_diets = diet_mb[diet_mb > 0]
    foods_upper_bounds = positive_diets.quantile(0.95).fillna(0.0)
    foods_upper_bounds = foods_upper_bounds[all_features]
    foods_lower_bounds = positive_diets.quantile(0.05).fillna(0.0)
    foods_lower_bounds = foods_lower_bounds[all_features]

    # -----------------------------
    # Load scalers
    # -----------------------------
    with open(home_path + f"data/{SPECIES}/diet_scaler{suffix_foods}.pkl", "rb") as f:
        diet_scaler = pickle.load(f)
    with open(home_path + f"data/{SPECIES}/age_scaler{suffix_foods}.pkl", "rb") as f:
        age_scaler = pickle.load(f)
    with open(home_path + f"data/{SPECIES}/mb_scaler{suffix_foods}.pkl", "rb") as f:
        mb_scaler = pickle.load(f)

    warnings.filterwarnings("ignore", message="Trying to unpickle estimator StandardScaler")

    alcoholic_features = {
        "Beer", "Campari", "Cocktail", "Dessert Wine", "Gin and tonic", "Light Beer",
        "Malt beverage", "Ouzo", "Sweet wine", "Vodka or Arak", "Whiskey", "Wine"
    }

    # -----------------------------
    # Subject extraction
    # -----------------------------
    def get_subject(diet_mb_df: pd.DataFrame, i: int):
        subject = diet_mb_df.iloc[i]
        subject_gender = subject["sex"]
        subject_age = subject["age"]

        subject_diet = subject[all_features + ["Energy"]]
        subject_calories = subject_diet["Energy"]

        # Drop age/sex from diet vector; they’re passed separately
        subject_diet = subject_diet[~subject_diet.index.isin(["age", "sex"])]
        subject_mb = subject[targets].copy()

        return subject_diet, subject_mb, subject_calories, subject_gender, subject_age

    # -----------------------------
    # Core prediction helpers (same as yours, minimally cleaned)
    # -----------------------------
    def normalize_predicted_microbiome_optimized(
        predicted_mb_log_z: pd.DataFrame,
        mb_scaler_obj: StandardScaler,
        non_existing_mask: np.ndarray,
        targets_list: list,
    ):
        baseline = predicted_mb_log_z.iloc[:, :2]
        z_arr = predicted_mb_log_z[targets_list].values.reshape(-1)

        inv_log10 = mb_scaler_obj.inverse_transform(z_arr.reshape(1, -1))[0]
        lin = 10 ** inv_log10

        lin[non_existing_mask] = 1e-4

        existing = ~non_existing_mask
        lin[existing] /= lin[existing].sum()

        log10_arr = np.log10(lin)
        norm_df = pd.DataFrame([log10_arr], columns=targets_list, index=predicted_mb_log_z.index)
        return pd.concat([baseline, norm_df], axis=1)

    def check_constraints(
        candidate: pd.Series,
        original: pd.Series,
        kcal: float,
        gender: str,
        max_kcal_pct: float,
        alcohol_idx: np.ndarray,
    ) -> bool:
        # total kcal change
        delta_kcal = (candidate - original) * kcal
        if abs(delta_kcal.sum()) > kcal * max_kcal_pct:
            return False

        # alcohol %
        if candidate[alcohol_idx].sum() > (0.05 if gender == "female" else 0.07):
            return False

        return True

    def predict_microbiome_from_diet_optimized(
        candidate_all: pd.Series,
        model_map: dict,
        diet_scaler_obj: StandardScaler,
        mb_scaler_obj: StandardScaler,
        age_scaler_obj: StandardScaler,
        subject_age_val: float,
        subject_gender_val,
        all_features_list: list,
        diet_features_list: list,
        targets_list: list,
    ) -> pd.DataFrame:
        vec = pd.Series(index=all_features_list, dtype=float)

        vec[diet_features_list] = diet_scaler_obj.transform([candidate_all[diet_features_list]])[0]
        vec["age"] = age_scaler_obj.transform([[subject_age_val]])[0, 0]
        vec["sex"] = subject_gender_val

        df_in = pd.DataFrame([vec.values], columns=all_features_list)

        preds = [model_map[i].predict(df_in)[0] for i in range(len(targets_list))]
        z_arr = np.array(preds)

        meta = pd.DataFrame([[vec["age"], vec["sex"]]], columns=["age", "sex"])
        z_df = pd.DataFrame([z_arr], columns=targets_list)
        return pd.concat([meta, z_df], axis=1)

    def predict_triglycerides(
        subject_mb_df: pd.DataFrame,
        targets_list: list,
        mb_scaler_obj: StandardScaler,
        age_scaler_obj: StandardScaler,
        subject_age_val: float,
        subject_gender_val,
        tg_model,
    ) -> float:
        mb_scaled = mb_scaler_obj.transform(subject_mb_df[targets_list])
        df = pd.DataFrame(mb_scaled, columns=targets_list, index=subject_mb_df.index)

        df["age"] = age_scaler_obj.transform([[subject_age_val]])[0, 0]
        df["sex"] = subject_gender_val

        ordered = df[["age", "sex"] + targets_list]
        return tg_model.predict(ordered)[0]

    # ============================================================
    # GREEDY OPTIMIZER (WITH NEGATIVE CONTROL SWITCH)
    # ============================================================
    def recommend_diet_change_greedy(
        subject_id: int,
        subject_diet: pd.Series,
        subject_mb: pd.Series,
        all_features_list: list,
        diet_scaler_obj: StandardScaler,
        mb_scaler_obj: StandardScaler,
        age_scaler_obj: StandardScaler,
        model_diet_to_mb_path: str,
        model_tg_path: str,
        nova_df: pd.DataFrame,
        food_upper_bounds_ser: pd.Series,
        food_lower_bounds_ser: pd.Series,
        subject_calories: float,
        subject_gender: str,
        subject_age: float,
        targets_list: list,
        max_total_kcal_pct_change: float,
        max_iter: int,
        negative_control: bool = False,  # <<< key flag
    ):
        # --- LOAD MODELS ON WORKER NODE ---
        with open(model_diet_to_mb_path, "rb") as f:
            model_diet_to_mb = pickle.load(f)

        with open(model_tg_path, "rb") as f:
            phenotypes_models_dict = pickle.load(f)
        model_tg = phenotypes_models_dict[PHENOTYPE]
        # -----------------------------------

        orig_pct = subject_diet[food_shortnames].copy()
        total_steps = pd.Series(0.0, index=orig_pct.index)
        used_steps = set()

        non_exist = np.isin(targets_list, subject_mb[subject_mb == -4.0].index)
        alcohol_idx = np.isin(orig_pct.index, list(alcoholic_features))

        # Baseline (diet -> microbiome -> TG)
        baseline_mb = predict_microbiome_from_diet_optimized(
            subject_diet,
            model_diet_to_mb,
            diet_scaler_obj,
            mb_scaler_obj,
            age_scaler_obj,
            subject_age,
            subject_gender,
            all_features_list,
            DIET_ONLY_FEATURES,
            targets_list,
        )
        baseline_norm_mb = normalize_predicted_microbiome_optimized(
            baseline_mb, mb_scaler_obj, non_exist, targets_list
        )

        baseline_tg = predict_triglycerides(
            baseline_norm_mb,
            targets_list,
            mb_scaler_obj,
            age_scaler_obj,
            subject_age,
            subject_gender,
            model_tg,
        )
        print("Baseline triglycerides:", baseline_tg)

        # ---- NEGATIVE CONTROL: freeze TG evaluation ----
        frozen_tg = float(baseline_tg)
        # -----------------------------------------------

        current_tg = float(baseline_tg)

        # greedy loop
        direction_phases = [(1, -1), (0.5, -0.5)]
        for phase, directions in enumerate(direction_phases, start=1):
            print(f"--- Phase {phase}: using directions {directions} ---")
            for it in range(max_iter):
                print(f"-------------- Iteration: {it + 1} --------------")
                best_pred = current_tg
                best_step = None

                for food in orig_pct.index[orig_pct > 0]:
                    if food not in food_upper_bounds_ser.index or food not in food_lower_bounds_ser.index:
                        continue

                    upper_bound = food_upper_bounds_ser[food]
                    lower_bound = food_lower_bounds_ser[food]
                    base = orig_pct[food]

                    # Skip foods outside percentile bounds
                    if base > upper_bound or base < lower_bound:
                        continue

                    for direction in directions:
                        raw = direction * base
                        current_val = orig_pct[food] + total_steps[food]

                        if direction > 0:
                            remaining_room = upper_bound - current_val
                            step_sz = min(raw, remaining_room)
                            if nova_df.get(food, 4) == 4 or remaining_room <= 0:
                                continue
                        else:
                            remaining_room = current_val
                            step_sz = -min(abs(raw), remaining_room)
                            if remaining_room <= 0:
                                continue

                        cand_pct = orig_pct + total_steps + step_sz * (orig_pct.index == food)
                        cand_pct = cand_pct / cand_pct.sum()

                        cand_all = subject_diet.copy()
                        cand_all.update(cand_pct)

                        if not check_constraints(
                            cand_pct,
                            orig_pct,
                            subject_calories,
                            subject_gender,
                            max_total_kcal_pct_change,
                            alcohol_idx,
                        ):
                            continue

                        # candidate microbiome
                        pred_mb = predict_microbiome_from_diet_optimized(
                            cand_all,
                            model_diet_to_mb,
                            diet_scaler_obj,
                            mb_scaler_obj,
                            age_scaler_obj,
                            subject_age,
                            subject_gender,
                            all_features_list,
                            DIET_ONLY_FEATURES,
                            targets_list,
                        )
                        norm_mb = normalize_predicted_microbiome_optimized(
                            pred_mb, mb_scaler_obj, non_exist, targets_list
                        )

                        # candidate TG
                        if negative_control:
                            # NEGATIVE CONTROL: TG does NOT depend on candidate changes
                            pred_tg = frozen_tg
                        else:
                            pred_tg = predict_triglycerides(
                                norm_mb,
                                targets_list,
                                mb_scaler_obj,
                                age_scaler_obj,
                                subject_age,
                                subject_gender,
                                model_tg,
                            )

                        if pred_tg < best_pred:
                            best_pred = pred_tg
                            best_step = (food, direction, step_sz)

                # no improvement
                if best_step is None:
                    break

                # apply best step
                food, direction, step_sz = best_step
                total_steps[food] += step_sz
                used_steps.add(f"{food}_{direction}")
                current_tg = best_pred

        iterations = it + 1 if "it" in locals() else 0

        # final diet
        final_pct = orig_pct + total_steps
        final_pct /= final_pct.sum()
        final_diet = subject_diet.copy()
        final_diet.update(final_pct)

        # final microbiome
        final_mb = predict_microbiome_from_diet_optimized(
            final_diet,
            model_diet_to_mb,
            diet_scaler_obj,
            mb_scaler_obj,
            age_scaler_obj,
            subject_age,
            subject_gender,
            all_features_list,
            DIET_ONLY_FEATURES,
            targets_list,
        )
        final_norm_mb = normalize_predicted_microbiome_optimized(
            final_mb, mb_scaler_obj, non_exist, targets_list
        )

        # final TG
        if negative_control:
            final_tg = frozen_tg
        else:
            final_tg = predict_triglycerides(
                final_norm_mb,
                targets_list,
                mb_scaler_obj,
                age_scaler_obj,
                subject_age,
                subject_gender,
                model_tg,
            )

        species_cols = [c for c in baseline_norm_mb.columns if c not in ("age", "sex")]
        baseline_lin_mb = 10 ** baseline_norm_mb[species_cols]
        final_lin_mb = 10 ** final_norm_mb[species_cols]

        return {
            "subject_id": subject_id,
            "negative_control": bool(negative_control),
            "original_diet": orig_pct,
            "recommended_diet": final_diet,
            "original_tg": float(baseline_tg),
            "final_tg": float(final_tg),
            "predicted_baseline_microbiome": baseline_lin_mb,
            "final_microbiome": final_lin_mb,
            "delta_microbiome": final_lin_mb.subtract(baseline_lin_mb, axis=1),
            "total_diet_change": total_steps,
            "steps_taken": int(len(used_steps)),
            "iterations": int(iterations),
            "Energy": float(subject_calories),
        }

    # ============================================================
    # QUEUE EXECUTION
    # ============================================================
    results = []
    param_methods = {}

    if foods_only:
        diet_species_model_path = (
            f"/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/"
            f"models/regression/{SPECIES}/models_LGBM_abundance_longitudinal{suffix_foods}.pkl"
        )
    else:
        diet_species_model_path = (
            "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/"
            "models/regression/segal_species/models_LGBM_abundance_longitudinal.pkl"
        )

    phenotypes_models_path = (
        "/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/"
        "models/segal_species/models_mb_phenotypes.pkl"
    )


    for i in range(2):
    # for i in range(len(diet_mb_test)):
        print(f"Submitting job for subject {i}...")

        subject_diet, subject_mb, subject_kcal, subject_sex, subject_age = get_subject(diet_mb, i)

        param_methods[i] = q.method(
            recommend_diet_change_greedy,
            (
                i,
                subject_diet,
                subject_mb,
                all_features_formatted,
                diet_scaler,
                mb_scaler,
                age_scaler,
                diet_species_model_path,
                phenotypes_models_path,
                nova_foods_df,
                foods_upper_bounds,
                foods_lower_bounds,
                subject_kcal,
                subject_sex,
                subject_age,
                targets,
                0.2,   # max_total_kcal_pct_change
                1,    # max_iter
                NEGATIVE_CONTROL,  # <<< pass flag
            ),
        )

    print("All jobs have been submitted to the queue.")

    for (param, stub_method) in param_methods.items():
        print(f"Waiting for result from subject {param}...")
        results.append(q.waitforresult(stub_method))
        print(f"Received result from subject {param}.")

    results_df = pd.DataFrame(results)

    out_suffix = "_negative_control" if NEGATIVE_CONTROL else ""
    out_path = home_path + f"data/diet_intervention_results_queue{suffix_foods}_{PHENOTYPE}{out_suffix}.pkl"
    results_df.to_pickle(out_path)
    print(f"SAVED: {out_path}")


# ============================================================
# RUN
# ============================================================
addloglevels.sethandlers()
os.chdir("/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/code")

with config.qp(
    jobname="lgbm",
    _delete_csh_withnoerr=True,
    q=["himem7.q"],
    _trds_def=8,
    max_u=200,
    _mem_def="1G",
) as q:
    q.startpermanentrun()
    main(q)

print("FINISHED")
